# Análise e Clusterização de Senhas Vazadas - PySpark / Databricks

Versão distribuída (PySpark) da análise de 14 milhões de senhas do dataset rockyou, processada no Databricks. Porta a análise originalmente feita em pandas/scikit-learn para o ambiente Spark, demonstrando processamento de dados em escala.

> Versão original em pandas: github.com/martlnove/analise-senhas-clusterizacao

## 1. Configuração do ambiente

Na Free Edition (serverless), o armazenamento é feito via Volumes do Unity Catalog. Primeiro criamos a estrutura para receber o dataset.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.senhas COMMENT 'Projeto analise de senhas'")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.senhas.dados")

print("Schema e volume prontos.")

Schema e volume prontos.


## 2. Upload do dataset

Faça o upload do arquivo `rockyou.txt` para o Volume criado, no caminho:

`/Volumes/workspace/senhas/dados/rockyou.txt`

In [0]:
ARQUIVO = "/Volumes/workspace/senhas/dados/rockyou.txt"

df = spark.read.text(ARQUIVO).withColumnRenamed("value", "senha")

print(f"Registros carregados: {df.count():,}")
df.show(5, truncate=False)

Registros carregados: 14,344,381
+---------+
|senha    |
+---------+
|123456   |
|12345    |
|123456789|
|password |
|iloveyou |
+---------+
only showing top 5 rows


## 3. Limpeza

Remoção de linhas nulas/vazias e de senhas duplicadas

In [0]:
from pyspark.sql.functions import col, trim, length

total_antes = df.count()

df = df.filter(col("senha").isNotNull())
df = df.filter(length(trim(col("senha"))) > 0)
df = df.dropDuplicates(["senha"])

total_depois = df.count()
print(f"Removidos (nulos + duplicatas): {total_antes - total_depois:,}")
print(f"Senhas únicas restantes       : {total_depois:,}")

Removidos (nulos + duplicatas): 10
Senhas únicas restantes       : 14,344,371


## 4. Feature Engineering

Extração de 9 atributos por senha.

In [0]:
from pyspark.sql.functions import (
    length, regexp_replace, when, col, expr
)

df = df.withColumn("tamanho", length("senha"))

# Contagens por tipo de caractere
df = df.withColumn("qtd_maiusculas", length("senha") - length(regexp_replace("senha", r"[A-Z]", "")))
df = df.withColumn("qtd_minusculas", length("senha") - length(regexp_replace("senha", r"[a-z]", "")))
df = df.withColumn("qtd_numeros",    length("senha") - length(regexp_replace("senha", r"[0-9]", "")))
# Especiais: tudo que não é letra nem número
df = df.withColumn("qtd_especiais",  length("senha") - length(regexp_replace("senha", r"[A-Za-z0-9]", "")))

# Flags binárias (tem ou não tem)
df = df.withColumn("tem_maiuscula", when(col("qtd_maiusculas") > 0, 1).otherwise(0))
df = df.withColumn("tem_minuscula", when(col("qtd_minusculas") > 0, 1).otherwise(0))
df = df.withColumn("tem_numero",    when(col("qtd_numeros")    > 0, 1).otherwise(0))
df = df.withColumn("tem_especial",  when(col("qtd_especiais")  > 0, 1).otherwise(0))

df.select("senha", "tamanho", "tem_maiuscula", "tem_minuscula",
          "tem_numero", "tem_especial").show(5, truncate=False)

+-------+-------+-------------+-------------+----------+------------+
|senha  |tamanho|tem_maiuscula|tem_minuscula|tem_numero|tem_especial|
+-------+-------+-------------+-------------+----------+------------+
|melissa|7      |0            |1            |0         |1           |
|159753 |6      |0            |0            |1         |1           |
|bonita |6      |0            |1            |0         |1           |
|fluffy |6      |0            |1            |0         |1           |
|159357 |6      |0            |0            |1         |1           |
+-------+-------+-------------+-------------+----------+------------+
only showing top 5 rows


## 5. Persistir como tabela Delta

Salvamos o DataFrame processado como tabela Delta. Isso evita reprocessar tudo nos próximos blocos.

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.senhas.senhas_features")

print("Tabela 'workspace.senhas.senhas_features' criada.")
print(f"Total de registros: {spark.table('workspace.senhas.senhas_features').count():,}")

Tabela 'workspace.senhas.senhas_features' criada.
Total de registros: 14,344,371
